# 09 · Statistics

**Reads the JSON outputs of notebooks 03, 06, 07 and 08. Computes nothing new
from raw data.**

One place where every number in the paper gets its interval, its test, and its
caveat. If a statistic appears in the write-up, it is produced here.

## What it does
- **Bootstrap CIs** on every headline metric, so a point estimate from 5 folds is
  never reported bare.
- **Paired tests** on the tissue pathway, with the non-significant case stated
  correctly: no evidence of a difference is not evidence of equivalence.
- **The gate test**, an independent second line of evidence on the same question.
- **Holm-Bonferroni** across the tissue-pathway family, because running two tests
  and reporting the better one is how false positives get published.
- **Carries the circularity flag through** from notebook 06, so the severity
  ablation cannot be read as "tissue features help" when the tissue input is the
  label-generating variable.

## Missing files are flagged, not skipped
Any absent source is listed in `missing_sources` and its section marked
unverifiable. That distinction matters for the verification log: a flagged gap is
honest, a silent omission is not.

## Output
`stats_report.json`, `infection_ci.csv`


In [ ]:
# Cell 1 · config and inputs
from pathlib import Path
import pandas as pd, numpy as np, json, warnings
warnings.filterwarnings('ignore')
from scipy import stats

INTERIM = Path('data/interim')
OUT     = Path('outputs'); OUT.mkdir(exist_ok=True)

SEED, N_BOOT = 42, 2000
rng = np.random.default_rng(SEED)

SOURCES = {
    'severity'  : OUT / 'results_severity.json',
    'infection' : OUT / 'results_infection.json',
    'gradcam'   : OUT / 'gradcam_scores.json',
    'validation': INTERIM / 'validation_report.json',
}

data, missing = {}, []
for k, p in SOURCES.items():
    if p.exists():
        data[k] = json.load(open(p))
        print(f'  loaded {p.name}')
    else:
        missing.append(k)
        print(f'  MISSING {p.name} -> {k} statistics will be FLAGGED')

if not data:
    print('\nSTOPPING. No result files found. Run notebooks 06-08 first.')
    raise SystemExit(1)

report = {'missing_sources': missing}

In [ ]:
# Cell 2 · bootstrap confidence intervals
# Percentile bootstrap over evaluation units. Reported for every headline
# metric, because a single point estimate from 5 folds invites the
# question "how stable is that" and this answers it before it is asked.
def boot_ci(values, stat=np.mean, n_boot=N_BOOT, alpha=0.05):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if len(v) < 2:
        return (float('nan'), float('nan'))
    idx = rng.integers(0, len(v), size=(n_boot, len(v)))
    dist = np.array([stat(v[i]) for i in idx])
    lo, hi = np.percentile(dist, [100*alpha/2, 100*(1-alpha/2)])
    return float(lo), float(hi)

def fmt_ci(point, lo, hi):
    if np.isnan(lo): return f'{point:.4f}  (CI undefined, too few folds)'
    return f'{point:.4f}  95% CI [{lo:.4f}, {hi:.4f}]'

print('bootstrap ready (percentile, {} resamples)'.format(N_BOOT))

In [ ]:
# Cell 3 · infection: per-fold spread and bootstrap CI
if 'infection' in data:
    print('INFECTION CONTROL EXPERIMENT')
    rows = []
    for cfg in data['infection']['configs']:
        pf = cfg.get('per_fold_auroc', [])
        lo, hi = boot_ci(pf)
        rows.append(dict(configuration=cfg['name'], auroc=cfg['auroc'],
                         ci_low=lo, ci_high=hi,
                         fold_sd=float(np.nanstd(pf)) if pf else float('nan'),
                         n_folds=int(np.sum(~np.isnan(pf))) if pf else 0))
        print(f"  {cfg['name']:<28} {fmt_ci(cfg['auroc'], lo, hi)}")
        if pf:
            print(f"    per fold {[round(a,4) for a in pf]}  "
                  f"sd {np.nanstd(pf):.4f}")
    inf_tab = pd.DataFrame(rows)
    report['infection'] = rows
else:
    print('INFECTION — FLAGGED: results_infection.json not found')
    inf_tab = None

In [ ]:
# Cell 4 · the tissue-pathway test, stated carefully
# The question is not "is CNN-only better" but "is there any evidence the
# tissue pathway helps". A non-significant difference supports neither
# model being better; it does not prove equivalence. Both readings are
# printed so the write-up cannot overclaim in either direction.
if 'infection' in data and len(data['infection']['configs']) > 1:
    cfgs = {c['name']: c for c in data['infection']['configs']}
    base = cfgs.get('CNN only')
    print('\nTISSUE PATHWAY TEST (infection)')
    if base and 'per_fold_auroc' in base:
        b = np.array(base['per_fold_auroc'], dtype=float)
        for name, c in cfgs.items():
            if name == 'CNN only' or 'per_fold_auroc' not in c: continue
            o = np.array(c['per_fold_auroc'], dtype=float)
            ok = ~(np.isnan(b) | np.isnan(o))
            if ok.sum() < 2:
                print(f'  {name}: fewer than 2 usable folds, skipped')
                continue
            d = o[ok] - b[ok]
            t, p = stats.ttest_rel(o[ok], b[ok])
            lo, hi = boot_ci(d)
            print(f'  {name} minus CNN only')
            print(f'    mean {d.mean():+.4f}   {fmt_ci(d.mean(), lo, hi)}')
            print(f'    paired t p = {p:.3f}   '
                  f'({"no evidence of a difference" if p > 0.05 else "difference detected"})')
            if p > 0.05:
                print('    Reading: this does not establish equivalence, only')
                print('    that 5 folds cannot resolve a difference this small.')
            report.setdefault('tissue_test', []).append(
                dict(comparison=name, mean_diff=float(d.mean()),
                     p_value=float(p), ci=[lo, hi], n_folds=int(ok.sum())))
    # the gate is the second, independent line of evidence
    tgo = next((c for c in cfgs.values() if 'gated' in c['name']), None)
    if tgo and 'gamma_per_fold' in tgo:
        g = np.array(tgo['gamma_per_fold'], dtype=float)
        t1, p1 = stats.ttest_1samp(g, 0.0)
        print(f'\n  learned gate gamma vs zero: mean {g.mean():+.4f}, '
              f'sd {g.std():.4f}, p = {p1:.3f}')
        signs = np.sign(g)
        print(f'    sign flips across folds: '
              f'{int((signs[:-1] != signs[1:]).sum())} of {len(g)-1} transitions')
        if p1 > 0.05:
            print('    gamma is not distinguishable from zero: the model did')
            print('    not settle on a consistent use of the tissue input.')
        report['gate_test'] = dict(mean=float(g.mean()), sd=float(g.std()),
                                   p_value=float(p1),
                                   per_fold=[float(x) for x in g])

In [ ]:
# Cell 5 · severity: the ablation, with the circularity caveat carried through
if 'severity' in data:
    print('\nSEVERITY (derived labels)')
    cfgs = data['severity']['configs']
    base = next((c for c in cfgs if c['name'] == 'CNN only (CORAL)'), None)
    for c in cfgs:
        line = f"  {c['name']:<28} QWK {c['qwk']:.4f}"
        if base and c is not base:
            line += f"   delta {c['qwk'] - base['qwk']:+.4f}"
        print(line)
        if c.get('interpretation_flag'):
            print(f"      FLAG: {c['interpretation_flag']}")
    # mild is unmeasurable; state the sample size rather than a metric
    n_mild = data['severity'].get('class_counts', {}).get('mild')
    if n_mild is not None:
        print(f"\n  mild class: {n_mild} photographs total across the corpus.")
        print(f'  Any per-class metric on mild has an interval too wide to')
        print(f'  support a claim; the severe-vs-rest column is the')
        print(f'  measurable result.')
    report['severity'] = [
        dict(name=c['name'], qwk=c['qwk'],
             flag=c.get('interpretation_flag')) for c in cfgs]
else:
    print('\nSEVERITY — FLAGGED: results_severity.json not found')

In [ ]:
# Cell 6 · the control comparison, as a single defensible statement
# This is the sentence the paper's argument rests on, so it is assembled
# from the stored numbers rather than typed by hand.
print('\n' + '=' * 62)
print('CONTROL COMPARISON')
print('=' * 62)
if 'severity' in data and 'infection' in data:
    sev_best = max(data['severity']['configs'],
                   key=lambda c: c.get('severe_vs_rest_f1', 0))
    # exclude tissue-fed rows: they are circular by construction
    sev_cnn = [c for c in data['severity']['configs']
               if c['name'].startswith('CNN only')]
    sev_pixel = max(sev_cnn, key=lambda c: c['qwk']) if sev_cnn else None
    inf_best = max(data['infection']['configs'], key=lambda c: c['auroc'])

    print('  From pixels alone, identical pipeline, different labels:')
    if sev_pixel:
        print(f"    derived colour labels  QWK {sev_pixel['qwk']:.4f}   "
              f"({sev_pixel['name']})")
    print(f"    expert infection labels AUROC {inf_best['auroc']:.4f}   "
          f"({inf_best['name']})")
    print()
    print('  The architectures, preprocessing, grouping and code are')
    print('  unchanged between these rows. Only the label source differs.')
    print('  That isolates label provenance and rules out model capacity,')
    print('  input resolution and training configuration as explanations.')
    report['control_comparison'] = dict(
        severity_qwk_pixels_only=sev_pixel['qwk'] if sev_pixel else None,
        infection_auroc=inf_best['auroc'])
else:
    have = [k for k in ('severity','infection') if k in data]
    print(f'  FLAGGED: needs both severity and infection results; have {have}')

In [ ]:
# Cell 7 · label validation numbers, restated for the paper
if 'validation' in data:
    v = data['validation']
    print('\n' + '=' * 62)
    print('LABEL VALIDATION (notebook 03)')
    print('=' * 62)
    for key, title in [('check1_expert','expert agreement'),
                       ('check2_negctrl','negative control'),
                       ('check3_stability','stability under augmentation'),
                       ('check4_plausibility','corpus plausibility')]:
        if key not in v: continue
        c = v[key]
        st = c.get('status','?')
        print(f'  [{st.upper():<9}] {title}')
        for k2, val in c.items():
            if k2 in ('status','reason','note'): continue
            if isinstance(val, float): print(f'      {k2}: {val:.4f}')
            else: print(f'      {k2}: {val}')
        if c.get('reason'):  print(f'      reason: {c["reason"]}')
        if c.get('note'):    print(f'      note: {c["note"]}')
    report['validation'] = v
else:
    print('\nLABEL VALIDATION — FLAGGED: validation_report.json not found')

In [ ]:
# Cell 8 · multiple-comparison note
# Several tests are run across this project. Reporting them without
# acknowledging that inflates the chance one looks significant by luck.
# Holm-Bonferroni is applied to the family of tissue-pathway tests, which
# is the only family where a false positive would change a conclusion.
tests = []
for t in report.get('tissue_test', []):
    tests.append((t['comparison'], t['p_value']))
if 'gate_test' in report:
    tests.append(('gate gamma vs zero', report['gate_test']['p_value']))

if tests:
    print('\n' + '=' * 62)
    print('MULTIPLE COMPARISONS (Holm-Bonferroni)')
    print('=' * 62)
    order = sorted(range(len(tests)), key=lambda i: tests[i][1])
    m = len(tests)
    adj, prev = {}, 0.0
    for rank, i in enumerate(order):
        name, p = tests[i]
        a = min(1.0, max(prev, (m - rank) * p))
        adj[name] = a; prev = a
        print(f'  {name:<32} raw p {p:.3f}  ->  adjusted {a:.3f}'
              f'{"  significant" if a < 0.05 else ""}')
    report['holm_adjusted'] = adj
    if all(a >= 0.05 for a in adj.values()):
        print('\n  No test survives adjustment. The tissue pathway shows no')
        print('  effect by either the ablation or the gate, and this holds')
        print('  after accounting for running both.')

In [ ]:
# Cell 9 · save and summarise
with open(OUT / 'stats_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=float)
print(f'\nwrote {(OUT / "stats_report.json").resolve()}')
if inf_tab is not None:
    inf_tab.to_csv(OUT / 'infection_ci.csv', index=False)
    print(f'wrote {(OUT / "infection_ci.csv").resolve()}')

print('\n' + '=' * 58)
print('STAGE 09 COMPLETE')
print('=' * 58)
if report['missing_sources']:
    print(f'  FLAGGED sources: {report["missing_sources"]}')
    print('  Those sections are marked unverifiable rather than omitted.')
else:
    print('  all four source files present')
print('  every reported statistic traces to a stored result file')
print('\nnext: 10_figures.ipynb')